In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [3]:
# 1. 하수관거 및 부대시설 데이터 불러오기
sewer = pd.read_csv("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/하수관거_20260527133919.csv")
facilities = pd.read_csv("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/하수도+및+부대시설+현황_20260527133737.csv")

In [4]:
# 하수관거, 부대시설 파일에는 2023년 단일 데이터만 있으므로 2023년 데이터 활용

# 2. 필요한 컬럼 추출
s2 = sewer[['자치구별(2)', '2023.1', '2023.17']].copy()
s2.columns = ['자치구', 'SL_district', 'SF_district']
s2 = s2[~s2['자치구'].isin(['자치구별(2)', '소계'])]

In [5]:
fac = facilities[['자치구별(2)', '2023.5', '2023.6']].copy()
fac.columns = ['자치구', 'Manholes', 'CatchBasins']
fac = fac[~fac['자치구'].isin(['자치구별(2)', '소계'])]

In [6]:
# 결측치('-')를 0으로 치환하고 실수형으로 변환
for col in ['SL_district', 'SF_district']:
    s2[col] = s2[col].astype(str).str.replace(',', '').replace('-', '0').astype(float)
    
for col in ['Manholes', 'CatchBasins']:
    fac[col] = fac[col].astype(str).str.replace(',', '').replace('-', '0').astype(float)

In [7]:
# 자치구 기준으로 데이터프레임 병합
df = pd.merge(s2, fac, on='자치구', how='inner')

In [8]:
df.head()

,자치구,SL_district,SF_district,Manholes,CatchBasins
0,종로구,363920.0,0.0,10126.0,16383.0
1,중구,272027.0,0.0,8304.0,15045.0
2,용산구,375865.0,0.0,9970.0,17854.0
3,성동구,307722.0,355.0,8694.0,19332.0
4,광진구,370605.0,138.0,10693.0,25505.0


In [9]:
#  자치구 용량(C_gu) 연산
df['C_gu'] = df['SL_district'] * (1 + (df['SF_district'] / df['SL_district']).fillna(0))

In [10]:
# 관거 1m당 미세 배수 인프라 밀도 연산
df['MicroDrain_Density'] = (df['Manholes'] + df['CatchBasins']) / df['SL_district']

In [11]:
# 4. 역 정규화 (값이 작을수록 1에 가까워지도록 처리)
# 공식: (Max - X) / (Max - Min)
C_max = df['C_gu'].max()
C_min = df['C_gu'].min()
df['S2_Vulnerability'] = (C_max - df['C_gu']) / (C_max - C_min)

D_max = df['MicroDrain_Density'].max()
D_min = df['MicroDrain_Density'].min()
df['S3_Vulnerability'] = (D_max - df['MicroDrain_Density']) / (D_max - D_min)

In [12]:
df[['자치구', 'C_gu', 'MicroDrain_Density', 'S2_Vulnerability', 'S3_Vulnerability']].head()

,자치구,C_gu,MicroDrain_Density,S2_Vulnerability,S3_Vulnerability
0,종로구,363920.0,0.072843,0.788312,0.678981
1,중구,272027.0,0.085833,0.981168,0.323752
2,용산구,375865.0,0.074027,0.763243,0.646614
3,성동구,308077.0,0.091076,0.905510,0.180398
4,광진구,370743.0,0.097673,0.773993,0.000000


### Column 명
- C_gu : 하수처리 용량 원시값
    - 자치구별 하수관거 전체 길이(SL)와 우수관거(빗물관, SF)의 비율
    - 절대적인 하수처리 인프라 용량
    - 숫자가 클수록 빗물을 감당할 파이프라인이 튼튼한 것
- MicroDrain_Density : 배수 인프라 밀도 
    - 하수관거 1m 당 빗물받이와 맨홀의 개수
    - 수치가 클수록 개수가 많은 것
- S2_Vulnerability : 하수처리 미흡 지표 (산출식에서 S2에 해당)
    - C_gu를 0과 1 사이로 변환한 지표
    - 1에 가까울수록 하수처리가 부실하여 홍수에 취약함을 나타냄
- S3_Vulnerability : 배수 취약성
    - MicroDrain_Density를 변환한 값
    - 1에 가까울수록 빗물받이와 맨홀이 부족한 것

In [13]:
# 수치가 높은 자치구
# 2. S2_Vulnerability 기준 내림차순 정렬 및 출력

s2_sorted = df.sort_values(by='S2_Vulnerability', ascending=False)
print(s2_sorted[['자치구', 'S2_Vulnerability']].to_string(index=False))

 자치구  S2_Vulnerability
 금천구          1.000000
  중구          0.981168
 성동구          0.905510
 도봉구          0.859588
 동작구          0.852763
서대문구          0.820286
 강북구          0.803297
 종로구          0.788312
 광진구          0.773993
 용산구          0.763243
 양천구          0.723849
동대문구          0.702108
 구로구          0.696507
 관악구          0.644480
 중랑구          0.644362
 노원구          0.600800
 마포구          0.578654
 은평구          0.545073
 강동구          0.542032
영등포구          0.513655
 성북구          0.490007
 강서구          0.322564
 서초구          0.282198
 송파구          0.054136
 강남구          0.000000


In [14]:
# 3. S3_Vulnerability 기준 내림차순 정렬 및 출력

s3_sorted = df.sort_values(by='S3_Vulnerability', ascending=False)
print(s3_sorted[['자치구', 'S3_Vulnerability']].to_string(index=False))

 자치구  S3_Vulnerability
 송파구          1.000000
 노원구          0.903034
영등포구          0.875761
 강서구          0.805574
 구로구          0.805546
서대문구          0.790431
 강동구          0.774869
 강남구          0.770660
 종로구          0.678981
 용산구          0.646614
 서초구          0.605782
 은평구          0.520503
 마포구          0.512474
 도봉구          0.500024
 성북구          0.395440
 강북구          0.384733
 양천구          0.377645
 금천구          0.336729
  중구          0.323752
동대문구          0.252254
 중랑구          0.239697
 동작구          0.191674
 성동구          0.180398
 관악구          0.164657
 광진구          0.000000


### 1. 하수도 인프라 방어 지표 산출
- 빗물을 감당하는 관의 크기와 유입구의 수를 홍수 방어 지표로 연산
    - 하수처리 용량(C_gu) : 자치구별 하수관거 전체 길이와 우수관거(빗물관) 비율을 곱하여 산출한 절대적인 하수처리 용량
    - 배수 인프라 밀도(MicroDrain_Density) : 하수관거 1m 당 설치된 맨홀과 빗물받이의 총 개수

### 2. 위치 기반 매핑 및 데이터 병합
- 데이터에 포함된 자치구 기준으로 데이터 병합
- 결측치 0으로 치환, 문자열->실수형으로 변환하는 정제 과정 수행

### 3. 취약성 평가 지수
- 인프라 용량과 밀도가 낮을수록 홍수에 위험하므로, 작을수록 1에 가까워지도록 역정규화 수행
    - 하수처리 미흡 지표(S2) : 하수처리 용량 변환값, 1에 가까울수록 하수처리 부실
    - 배수 취약성 지표(S3) : 배수 인프라 밀도 변환값, 1에 가까울수록 부족함을 의미